# FinGPT Two-Agent Signal Pipeline (Colab)

## Cell 1 — GPU check & install

This cell verifies GPU availability and installs notebook dependencies for the full pipeline demo.

In [ ]:
# Check GPU
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install dependencies
!pip install -q transformers accelerate vllm alpaca-trade-api yfinance pydantic python-dotenv tqdm

## Cell 2 — Clone repo and set path

This cell clones the repository in Colab and sets the working directory so project imports resolve correctly.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

project_path = '/content/drive/MyDrive/FinGPT_Project'
repo_url = 'https://github.com/juankim834/FinGPT_Project.git'

if os.path.exists(project_path):
    print(f"Found repo at: {project_path}, updating")
    %cd {project_path}
    !git pull
else:
    print("Repo is not exist, try to clone intially")
    %cd /content/drive/MyDrive
    !git clone {repo_url}
    %cd {project_path}

# Shared paths on Google Drive for demo artifacts.
DEMO_OUTPUT_DIR = os.path.join(project_path, "output")
ARTICLE_CACHE_DIR = os.path.join(project_path, "cache")
DIAG_MD_DIR = os.path.join(DEMO_OUTPUT_DIR, "diagnostics_md")
os.makedirs(DEMO_OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTICLE_CACHE_DIR, exist_ok=True)
os.makedirs(DIAG_MD_DIR, exist_ok=True)
print(f"Drive output dir: {DEMO_OUTPUT_DIR}")
print(f"Article cache dir: {ARTICLE_CACHE_DIR}")
print(f"Diagnostics markdown dir: {DIAG_MD_DIR}")

## Cell 3 — Load secrets

This cell loads Alpaca credentials from Colab Secrets.

In [ ]:
import os
from google.colab import userdata

# Choose article provider: "finnhub" or "alpaca"
os.environ["NEWS_PROVIDER"] = "finnhub"

# Finnhub credential (required when NEWS_PROVIDER=finnhub)
os.environ["FINNHUB_API_KEY"] = userdata.get("FINNHUB_API_KEY")

# Alpaca credentials (optional unless NEWS_PROVIDER=alpaca)
os.environ["ALPACA_API_KEY"] = userdata.get("ALPACA_API_KEY")
os.environ["ALPACA_API_SECRET"] = userdata.get("ALPACA_API_SECRET")

# Resolve local merged model directory for vLLM loading.
model_path_candidates = [
    "/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm",
    "/content/MyDrive/deepseek_fingpt_outputs/merged_for_vllm",
]
resolved_model_path = None
for candidate in model_path_candidates:
    if os.path.isfile(os.path.join(candidate, "config.json")):
        resolved_model_path = candidate
        break

if resolved_model_path is None:
    raise FileNotFoundError(
        "Could not find local model folder with config.json. Checked: "
        + ", ".join(model_path_candidates)
    )

os.environ["FINGPT_MODEL_PATH"] = resolved_model_path

# Enable shared single-LLM mode for Agent 1 + Agent 2
os.environ["SHARE_SINGLE_LLM_BETWEEN_AGENTS"] = "true"

# Save raw markdown outputs from Agent 1/2 for debugging.
os.environ["FINGPT_DIAG_MD_DIR"] = globals().get(
    "DIAG_MD_DIR",
    os.path.join("output", "diagnostics_md"),
)

print("Colab secrets loaded and single-LLM mode configured.")
print(f"NEWS_PROVIDER={os.environ['NEWS_PROVIDER']}")
print(f"FINGPT_MODEL_PATH={os.environ['FINGPT_MODEL_PATH']}")
print(f"FINGPT_DIAG_MD_DIR={os.environ['FINGPT_DIAG_MD_DIR']}")

## Cell 4 — Load one shared model on vLLM and check VRAM

This cell loads one shared vLLM engine using the `FINGPT_MODEL_PATH` model ID and reports GPU memory before/after engine initialization.

In [ ]:
import os
import torch
from vllm import LLM, SamplingParams


def used_vram_gb() -> float:
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1e9


def gpu_supports_bf16() -> bool:
    if not torch.cuda.is_available():
        return False
    major, _minor = torch.cuda.get_device_capability(0)
    # Ampere+ generally supports fast BF16; older cards (e.g., T4) should use FP16.
    return major >= 8


if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this vLLM demo.")

print(f"VRAM before vLLM load: {used_vram_gb():.2f} GB")

model_id = os.environ.get("FINGPT_MODEL_PATH", "deepseek-ai/DeepSeek-R1-Distill-Llama-8B")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
prefer_dtype = "bfloat16" if gpu_supports_bf16() else "float16"

print(f"GPU detected: {gpu_name} ({total_vram_gb:.1f} GB)")
print(f"Model path: {model_id}")
print(f"Preferred dtype: {prefer_dtype}")

# Try safer startup configs first for Colab stability.
startup_profiles = [
    {"dtype": prefer_dtype, "gpu_memory_utilization": 0.85, "enforce_eager": True},
    {"dtype": "float16", "gpu_memory_utilization": 0.80, "enforce_eager": True},
]

llm = None
last_error = None
for i, profile in enumerate(startup_profiles, start=1):
    try:
        print(f"Attempt {i}: {profile}")
        llm = LLM(
            model=model_id,
            trust_remote_code=True,
            disable_log_stats=True,
            **profile,
        )
        break
    except Exception as exc:
        last_error = exc
        print(f"Attempt {i} failed: {type(exc).__name__}: {exc}")

if llm is None:
    raise RuntimeError(
        "vLLM failed to initialize after fallback attempts. "
        "Common causes: insufficient VRAM, incompatible dtype, or incomplete model files."
    ) from last_error

# Inject shared vLLM engine for Agent 1 and Agent 2.
from agent1.extractor import set_shared_vllm_engine as set_agent1_vllm_engine
from agent2.reasoner import set_shared_vllm_engine as set_agent2_vllm_engine

set_agent1_vllm_engine(llm)
set_agent2_vllm_engine(llm)

# Warm up once so memory usage reflects an initialized inference path.
_ = llm.generate(["Warmup."], SamplingParams(max_tokens=1, temperature=0.0))

used_after_load = used_vram_gb()
fits_device = used_after_load < total_vram_gb
fits_40gb = used_after_load <= 40.0

print(f"VRAM after vLLM load: {used_after_load:.2f} GB")
print(f"vLLM engine ready for model: {model_id}")
print(f"Fits current GPU capacity ({total_vram_gb:.1f} GB): {fits_device}")
print(f"Fits within 40 GB budget: {fits_40gb}")

## Cell 5 — Backtest-only: load FinGPT test set

This notebook runs only the backtest workflow using the test split of the FinGPT forecaster dataset.

Dataset source: `FinGPT/fingpt-forecaster-dow30-202305-202405`.

The test parquet file must be present in the project directory or on Google Drive. Path is configurable in the next cell.

In [ ]:
from backtest.dataset_parser import load_dataset, build_backtest_rows
import pandas as pd

DATASET_PATH = os.path.join(project_path, "data", "fingpt_dow30_test.parquet")
MAX_ROWS = None  # set an int for a quick smoke test

# Persist yfinance returns cache on Drive across notebook reruns.
os.environ["FINGPT_YF_CACHE_PATH"] = os.path.join(
    globals().get("DEMO_OUTPUT_DIR", "output"),
    "yfinance_return_cache.json",
)

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Test dataset not found: {DATASET_PATH}")

df_raw = load_dataset(DATASET_PATH)
backtest_rows = build_backtest_rows(df_raw)
if MAX_ROWS is not None:
    backtest_rows = backtest_rows[:MAX_ROWS]

print(f"Loaded {len(backtest_rows)} test rows for backtesting.")
print(f"yfinance cache file: {os.environ['FINGPT_YF_CACHE_PATH']}")
preview = pd.DataFrame([
    {
        "ticker": r["ticker"],
        "start_date": r["start_date"],
        "end_date": r["end_date"],
        "fingpt_label": r["fingpt_label"],
        "article_text": r["article_text"][:80] + "...",
    }
    for r in backtest_rows
])
display(preview.head(10))

## Cell 6 — Backtest: run news2signal pipeline

This cell feeds each test-set row through Agent 1 → Agent 2 and fetches realized returns via yfinance.

Repeated ticker/date requests use the yfinance cache configured above.

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from backtest.backtester import run_backtest
from backtest.price_fetcher import get_cache_stats

output_dir = globals().get("DEMO_OUTPUT_DIR", "output")
os.makedirs(output_dir, exist_ok=True)
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backtest_outpath = os.path.join(output_dir, f"backtest_testset_{ts}.csv")

results_df = run_backtest(
    dataset_path=DATASET_PATH,
    output_path=backtest_outpath,
    max_rows=MAX_ROWS,
)

print(f"Saved {len(results_df)} rows to {backtest_outpath}")
successful = results_df[results_df["skipped_reason"] == ""]
print(f"Successful: {len(successful)} / {len(results_df)}")

cache_stats = get_cache_stats()
print(f"yfinance cache entries (disk): {cache_stats['disk_entries']}")
print(f"yfinance cache entries (memory): {cache_stats['memory_entries']}")
display(results_df.head(10))

## Cell 7 — Backtest: metrics

This cell computes direction accuracy, Sharpe ratio, and alignment with the FinGPT baseline label.

In [ ]:
import json
from backtest.backtester import compute_metrics

metrics = compute_metrics(results_df)

print("=" * 48)
print("  Backtest Metrics")
print("=" * 48)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<30} {v:.4f}")
    else:
        print(f"  {k:<30} {v}")
print("=" * 48)

metrics_path = backtest_outpath.replace(".csv", "_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
print(f"Saved metrics to {metrics_path}")

## Cell 8 — Backtest: inspect skip reasons and cache file

This cell shows skip breakdown and confirms the yfinance cache file path.

In [ ]:
skip_counts = results_df["skipped_reason"].value_counts(dropna=False)
display(skip_counts.to_frame(name="count"))
print(f"yfinance cache file: {os.environ.get('FINGPT_YF_CACHE_PATH', 'output/yfinance_return_cache.json')}")

## Cell 9 — Release GPU

Unassign the runtime to free VRAM after the backtest run.

In [ ]:
from google.colab import runtime
runtime.unassign()